# マウナ・ロア気象観測所の$\mathsf{CO_2}$データ
- Python の vega_datasets というライブラリには，学習用の各種データセットが含まれている
- ここでは，その中からマウナ・ロア気象観測所の$\mathsf{CO_2}$データを時系列データとして利用する
- Date は日付 (西暦)，CO2 の単位は ppm (parts per million = 百万分率) である

※ このデータは局所的な大気変動の影響を受けにくい高高度で測定した大気中の$\mathsf{CO_2}$の  
　 データであって，マウナ・ロア山が排出する$\mathsf{CO_2}$のデータ**ではない!!**

※ 例によって，各コードの意味は理解しなくてもよい．何をしているのかをよく見ておくこと

In [ ]:
# vega_datasets ライブラリから data モジュールをインポートする
from vega_datasets import data

# data モジュールから目的のデータを取得して，変数 co2 に代入する
co2 = data.co2_concentration()

# co2 の表示
display(co2)

### データの概観

In [ ]:
# グラフの表示のために，plotly.express ライブラリをインポートする
import plotly.express as px

# co2 の内容をグラフで表示
plot = px.line(co2, x="Date", y="CO2")
display(plot)


### 前処理

In [ ]:
# co2 から 'Date', 'CO2' の列だけを取り出し，名前を付け替える
# ('adjusted CO2' という列が含まれていたことがあるので，'Date', 'CO2' だけ抽出)
src = co2[['Date', 'CO2']].rename(columns={"Date":"ds", "CO2":"y"})

# src の表示
display(src)

### 将来予測
- Prophet モジュール (prophet) を使って，co2 の将来の推移を予測する
- Prophet モジュールは，STAN というベイズ統計モデルを扱うシステムを利用している
  - ベイズ統計モデルは過去の観測値から将来の値を確率的に推定するモデルである

In [ ]:
# prophet ライブラリから Prophet モジュールをインポートする
from prophet import Prophet

# 新規に Prophet モデルを生成し，src にフィッティングさせる
model = Prophet(weekly_seasonality=False, daily_seasonality=False)
model.fit(src)

# model の ds を240か月延長したデータフレームを生成して future に代入する
future = model.make_future_dataframe(periods=240, freq="ME")

# future の表示
display(future)

In [ ]:
# future で延長した期間を model を用いて予測し，結果のデータフレームを forecast に代入する
forecast = model.predict(future)

# forecast の表示
display(forecast)

In [ ]:
# forecast の予測値 (yhat) のグラフを表示する
# 黒いプロットは観測値
# 色付きの範囲は予測の上下限を表す (デフォルトでは80%信頼区間)
_ = model.plot(forecast)

In [ ]:
# 傾向変動 (trend) と季節変動 (yearly) のグラフを表示する
_ = model.plot_components(forecast)

In [ ]:
# forecast のうち，値 0 の列と，yearly と同じ値の列を除き，並べ替えた列:
# (ds,   yhat,  yhat_lower, yhat_upper, trend,   trend_lower, trend_upper, yearly)
# (日付, 予測値, …の下限,    …の上限,     傾向変動, …の下限,     …の上限,      季節変動)
# の dataframe を生成して forecast2 に代入

forecast2 = forecast[["ds", "yhat", "yhat_lower", "yhat_upper", "trend", "trend_lower", "trend_upper", "yearly"]]

# 日付 (ds) と予測値 (yhat) の間に実測値 (y) の列を追加
forecast2.insert(1, "y", src["y"])

# forecast2 の先頭10行 (1年分(6,10月欠損)) と末尾12行 (1年分) の表示
display(forecast2.head(10))
display(forecast2.tail(12))

In [ ]:
# CSVファイルに保存

forecast2.to_csv("co2Prophet.csv", index=False)